# Playing with word2vec

In the bigram notebook we *trained* our own embeddings from scratch: a tiny model
predicted the next character, and 2-D character vectors fell out as a byproduct.
They self-organized — vowels clustered together, the start/end token sat off on its own.

**word2vec is the exact same idea, scaled up.** Same architecture (embedding lookup →
linear → softmax), same cross-entropy loss. The only differences:

- the vocabulary is **words**, not characters,
- the training pairs are `(center word, nearby word)` from a sliding window,
- it was trained on **billions** of words instead of a few thousand names.

Google trained one on ~100 billion words of Google News and released the vectors.
In this notebook we **download those pretrained vectors and play with them** — no
training required. The goal is to *see* what embeddings capture once they're trained
at scale: not spelling, but **meaning**.

In [ ]:
# One-time install (skip if you already have it)
# %pip install gensim

## 1. Download the model

`gensim` ships a downloader for popular pretrained embeddings. We'll grab the original
word2vec vectors Google released: **`word2vec-google-news-300`** — 3 million words/phrases,
each a **300-dimensional** vector.

> **Heads up:** the first run downloads ~1.6 GB and caches it under `~/gensim-data/`.
> Subsequent runs load instantly from cache.
>
> Want something lighter for a quick spin? Set `MODEL_NAME = "glove-wiki-gigaword-100"`
> (~130 MB). It's GloVe rather than word2vec, but the API and the ideas are identical.

In [ ]:
import gensim.downloader as api

MODEL_NAME = "word2vec-google-news-300"   # or "glove-wiki-gigaword-100" for a fast download
model = api.load(MODEL_NAME)
print(f"Loaded {MODEL_NAME!r}")

## 2. A trained model is just a big lookup table

Strip away the mystique and a word2vec model is a matrix of shape `(vocab_size, dim)` —
one row per word. That's the same `embeddings` matrix from the bigram notebook, only
much taller and wider. You look a word up and get its vector.

In [ ]:
print("vocabulary size:", len(model.index_to_key))
print("vector dimension:", model.vector_size)
print()

vec = model["king"]
print("type:", type(vec).__name__, "| shape:", vec.shape)
print("first 8 numbers of the 'king' vector:")
print(vec[:8])

These 300 numbers mean nothing on their own — no single dimension is "royalty" or
"gender." The meaning lives in the **geometry**: which vectors point in similar
directions. So before we compare *words*, let's get comfortable comparing *vectors*.

## 3. Vectors, distance, and similarity

A vector is just a list of numbers, and a list of numbers is a **point in space** (or,
equivalently, an arrow from the origin to that point). The `king` vector above is a point
in 300-dimensional space — impossible to picture. But every idea we need works the same in
any number of dimensions, so we'll build intuition in **2-D**, where we can draw it, then
apply it unchanged to the 300-D word vectors.

Here are two toy vectors as arrows from the origin. The two things we're about to measure
are drawn right on the picture: the **gap** between their tips (Euclidean distance) and the
**angle** between them (which cosine similarity is based on):

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

a = np.array([2.0, 1.0])
b = np.array([1.0, 2.0])

fig, ax = plt.subplots(figsize=(6, 6), dpi=130)

# the two vectors, drawn as arrows from the origin
for v, color, name in [(a, "#2c3e50", "a"), (b, "#c0392b", "b")]:
    ax.annotate("", xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2.5))
    ax.annotate(f"{name} = {v.tolist()}", v, color=color, fontsize=12,
                xytext=(6, 4), textcoords="offset points")

# Euclidean distance: the straight-line gap between the two tips
ax.plot([a[0], b[0]], [a[1], b[1]], "--", color="#7f8c8d", lw=1.5)
ax.annotate(f"distance ≈ {np.linalg.norm(a - b):.2f}", (a + b) / 2, color="#7f8c8d",
            fontsize=10, xytext=(8, 0), textcoords="offset points")

# Cosine: the angle between the two arrows, drawn as an arc near the origin
ang_a = np.degrees(np.arctan2(a[1], a[0]))
ang_b = np.degrees(np.arctan2(b[1], b[0]))
ax.add_patch(Arc((0, 0), 1.4, 1.4, theta1=ang_a, theta2=ang_b, color="#2980b9", lw=2))
mid = np.radians((ang_a + ang_b) / 2)
ax.annotate(f"{ang_b - ang_a:.1f}°", (0.95 * np.cos(mid), 0.95 * np.sin(mid)),
            color="#2980b9", fontsize=12, ha="center", va="center")

ax.set_xlim(-0.5, 3); ax.set_ylim(-0.5, 3); ax.set_aspect("equal")
ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
ax.grid(True, alpha=0.3)
ax.set_title("Two ways to compare vectors: the gap and the angle")
plt.show()

### Euclidean distance — the "ruler"

The most familiar way to compare two points: the straight-line distance between their
tips. It's the Pythagorean theorem generalized to any number of dimensions — square the
per-coordinate differences, sum them, take the square root:

$$\text{dist}(a, b) = \sqrt{\sum_i (a_i - b_i)^2} = \|a - b\|$$

Small distance = close together. `np.linalg.norm` computes it in one call.

In [ ]:
dist = np.linalg.norm(a - b)          # sqrt(sum((a - b)**2))
print("a - b        =", (a - b).tolist())
print("euclidean    =", round(float(dist), 3))

### Cosine similarity — the "angle"

Distance cares about the *gap* between the tips. Cosine similarity instead asks: **do the
two arrows point the same way?** It's the cosine of the angle between them:

$$\cos(a, b) = \frac{a \cdot b}{\|a\|\,\|b\|}$$

where $a \cdot b = \sum_i a_i b_i$ is the dot product. The result lives in $[-1, 1]$:

- **+1** → same direction (as similar as possible)
- **0** → perpendicular / unrelated
- **−1** → opposite direction

Crucially, dividing by the lengths $\|a\|\,\|b\|$ means cosine **ignores magnitude** —
only direction matters.

In [ ]:
cos = a.dot(b) / (np.linalg.norm(a) * np.linalg.norm(b))
angle = np.degrees(np.arccos(cos))
print("dot product  =", float(a.dot(b)))
print("cosine sim   =", round(float(cos), 3))
print("angle        =", round(float(angle), 1), "degrees")

### The key difference: magnitude

This is *the* reason embeddings prefer cosine. Consider three vectors:

- `p = [1, 1]` and `q = [100, 100]` point in the **exact same direction** — but their tips
  are far apart.
- `p = [1, 1]` and `r = [1, -1]` are the **same length** — but point at right angles.

Watch how the two measures disagree:

In [ ]:
p = np.array([1.0, 1.0])
q = np.array([100.0, 100.0])   # same direction as p, very different magnitude
r = np.array([1.0, -1.0])      # same magnitude as p, very different direction

def cosine(u, v):
    return float(u.dot(v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"p vs q:  euclidean = {np.linalg.norm(p - q):7.2f}   cosine = {cosine(p, q):.2f}")
print(f"p vs r:  euclidean = {np.linalg.norm(p - r):7.2f}   cosine = {cosine(p, r):.2f}")

`p` and `q` are a huge Euclidean distance apart, yet cosine says they're **identical**
(1.00) — because they point the same way. In word embeddings a vector's *magnitude* mostly
reflects things like word frequency, not meaning, so we usually want to ignore it. That's
why cosine is the default for comparing word vectors.

### How the two relate

They're not unrelated, though. If you **normalize** vectors to length 1 first, Euclidean
distance becomes a direct function of cosine similarity:

$$\|a - b\|^2 = 2\,\big(1 - \cos(a, b)\big)$$

So on unit-length vectors, "most similar by cosine" and "closest by Euclidean distance"
give the **same ranking**. Let's verify the identity numerically:

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)

an, bn = unit(a), unit(b)
lhs = np.linalg.norm(an - bn) ** 2
rhs = 2 * (1 - an.dot(bn))
print("||a - b||^2      =", round(float(lhs), 6))
print("2 * (1 - cos)    =", round(float(rhs), 6))   # identical

**Takeaway:** cosine similarity measures *direction* (angle), Euclidean distance
measures *direction and magnitude* (gap), and on normalized vectors they rank things
identically. `gensim` uses cosine under the hood — `most_similar` and `similarity` below
are exactly the `cos(a, b)` formula, just applied to 300-D word vectors instead of our 2-D
toys.

## 4. Nearest neighbors

The most direct way to probe an embedding space: **which words are closest to a given
word?** "Closest" here means highest **cosine similarity** — exactly the angle measure we
just built. Words used in similar contexts point the same way (cosine ≈ 1); unrelated words
point in unrelated directions (cosine ≈ 0).

`most_similar` ranks the whole vocabulary by cosine similarity for us.

In [ ]:
for word in ["king", "computer", "happy"]:
    neighbors = model.most_similar(word, topn=6)
    print(f"{word:>10} -> " + ", ".join(f"{w} ({s:.2f})" for w, s in neighbors))

### One word, one vector

Here's a limitation worth seeing early. word2vec gives every word **exactly one vector**,
no matter how many meanings it has. So an ambiguous word can't keep its senses apart — it
**collapses to whichever meaning dominated the training text**.

Take `mouse`. You might expect the animal, but this model was trained on ~2013 news, where
"mouse" overwhelmingly meant the computer peripheral — so that's the only sense you see:

In [ ]:
for w, s in model.most_similar("mouse", topn=8):
    print(f"  {w:<26} {s:.3f}")

Not a single rodent. The same happens with `apple` (all fruit, no tech company) and
`java` (all coffee, no programming language) — try them. A single vector simply can't hold
two meanings, so the dominant one wins.

This "one vector per word" limit is the core weakness of **static** embeddings, and it's
exactly what **contextual** embeddings fix — we come back to it at the end.

### Measuring similarity directly

`similarity(a, b)` gives the raw cosine between two words. Related words score high,
unrelated words score low — a quick sanity check that the geometry means something.

In [ ]:
pairs = [("cat", "dog"), ("cat", "car"), ("coffee", "tea"),
         ("coffee", "keyboard"), ("king", "queen"), ("king", "banana")]

for a, b in pairs:
    print(f"  {a:>7} ~ {b:<9} {model.similarity(a, b):.3f}")

## 5. The famous trick: analogies as arithmetic

Because directions in the space are meaningful, **relationships become vector offsets**.
The step from `man` to `woman` is roughly the same step as `king` to `queen` — a
"gender" direction. So you can literally do arithmetic on meaning:

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$

`most_similar(positive=[...], negative=[...])` computes exactly this: add the positive
vectors, subtract the negative ones, and find the nearest word to the result.

In [ ]:
def analogy(a, b, c, topn=1):
    """a is to b as c is to ?   ->   b - a + c"""
    res = model.most_similar(positive=[b, c], negative=[a], topn=topn)
    print(f"  {a} : {b}  ::  {c} : {res[0][0]}   (score {res[0][1]:.2f})")

analogy("man", "king", "woman")        # -> queen
analogy("paris", "france", "tokyo")    # -> japan   (capital : country)
analogy("walk", "walking", "swim")     # -> swimming (verb tense)
analogy("good", "better", "bad")       # -> worse    (comparative)
analogy("dog", "puppy", "cat")         # -> kitten   (animal : baby animal)

### Odd one out

`doesnt_match` picks the word whose vector sits furthest from the group's center — a
neat way to see clustering in action.

In [ ]:
groups = [
    "breakfast cereal lunch dinner",
    "france germany spain guitar",
    "red blue green happy",
]
for g in groups:
    print(f"  {g:<32} -> {model.doesnt_match(g.split())}")

## 6. Visualizing the space

The vectors live in 300 dimensions, so we can't look at them directly. We'll **project
to 2-D with PCA** (principal component analysis) — it finds the two directions along
which the words vary most and flattens onto that plane. Same spirit as the 2-D scatter
in the bigram notebook, but now the axes summarize 300 dimensions instead of 2.

We implement PCA in a few lines of NumPy via the SVD — no extra dependencies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def pca_2d(vectors):
    """Project (N, D) vectors to (N, 2) using PCA (via SVD)."""
    X = vectors - vectors.mean(axis=0)
    _, _, Vt = np.linalg.svd(X, full_matrices=False)
    return X @ Vt[:2].T


def plot_words(word_groups, title):
    """word_groups: dict of {group_name: [words]} -> colored 2-D scatter."""
    words = [w for group in word_groups.values() for w in group]
    vectors = np.stack([model[w] for w in words])
    coords = pca_2d(vectors)

    colors = plt.cm.tab10(np.linspace(0, 1, len(word_groups)))
    fig, ax = plt.subplots(figsize=(11, 8), dpi=150)

    i = 0
    for (name, group), color in zip(word_groups.items(), colors):
        for w in group:
            x, y = coords[i]
            ax.scatter(x, y, color=color, s=90, zorder=3)
            ax.annotate(w, (x, y), fontsize=11,
                        xytext=(5, 4), textcoords="offset points")
            i += 1
        ax.scatter([], [], color=color, s=90, label=name)  # legend entry

    ax.set_title(title, fontsize=14)
    ax.legend(loc="best", fontsize=10)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_words({
    "animals":   ["dog", "cat", "horse", "elephant", "lion", "tiger", "mouse"],
    "fruits":    ["apple", "banana", "orange", "grape", "mango", "peach"],
    "countries": ["france", "germany", "japan", "brazil", "canada", "egypt"],
    "colors":    ["red", "blue", "green", "yellow", "purple", "orange"],
}, title="word2vec groups words by meaning (PCA to 2-D)")

Words cluster by category with no labels ever given — purely from co-occurrence
statistics. (`orange` sits between fruits and colors, exactly as you'd hope for an
ambiguous word.)

### Relationships are directions

The analogy trick said `country - capital` is a consistent offset. If that's true, the
arrows from each **country to its capital** should be roughly **parallel**. Let's draw them.

In [ ]:
pairs = [("france", "paris"), ("germany", "berlin"), ("japan", "tokyo"),
         ("italy", "rome"), ("spain", "madrid"), ("russia", "moscow")]

words = [w for pair in pairs for w in pair]
coords = pca_2d(np.stack([model[w] for w in words]))

fig, ax = plt.subplots(figsize=(11, 8), dpi=150)
for i, (country, capital) in enumerate(pairs):
    c_xy, cap_xy = coords[2 * i], coords[2 * i + 1]
    ax.annotate("", xy=cap_xy, xytext=c_xy,
                arrowprops=dict(arrowstyle="->", color="#888", lw=1.5))
    for xy, w, col in [(c_xy, country, "#2c3e50"), (cap_xy, capital, "#c0392b")]:
        ax.scatter(*xy, color=col, s=80, zorder=3)
        ax.annotate(w, xy, fontsize=11, xytext=(5, 4), textcoords="offset points")

ax.set_title("country -> capital is a consistent direction", fontsize=14)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 7. A caveat: embeddings learn our biases

word2vec learns from human text, so it also absorbs human **stereotypes**. The same
analogy machinery that gives `king - man + woman = queen` produces uncomfortable results
too. This isn't a bug in the math — it's a faithful reflection of the training data, and
it's why practitioners audit and debias embeddings before deploying them.

In [ ]:
print("man : doctor  ::  woman : ",
      model.most_similar(positive=["doctor", "woman"], negative=["man"], topn=1)[0][0])
print("he  : doctor  ::  she   : ",
      model.most_similar(positive=["doctor", "she"], negative=["he"], topn=1)[0][0])

## 8. Where this goes next

word2vec gives each word **one fixed vector**. But "bank" means different things in
*river bank* and *bank account* — a single vector can't capture that. These are called
**static embeddings**.

The next leap is **contextual embeddings** (BERT, GPT, and friends): the vector for a
word is computed *from its sentence*, so "bank" gets a different vector each time. Same
core idea you've now seen twice — words become vectors, and geometry encodes meaning —
but the vectors become context-aware. That's where the rest of the book is headed.

### Recap

| | bigram notebook | word2vec |
|---|---|---|
| unit | characters | words |
| training pairs | `(char, next char)` | `(center, context)` in a window |
| architecture | lookup → linear → softmax | **the same** |
| loss | cross-entropy | cross-entropy (+ negative sampling) |
| scale | a few thousand names | ~100 billion words |
| what emerges | vowels vs. consonants | **meaning**: analogies, clusters, relations |
